# Working on linear regression with leave one out cross validation (LOOCV)

#### Generate synthetic dataset

In [22]:
from sklearn.datasets import make_regression
import pandas as pd
import numpy as np
def gen_data(n_samples=100, n_features=10, noise=5):
    X, y, theta_star = make_regression(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_features,
        noise=noise,
        coef=True,
        random_state=42
    )
    intercept = 7
    y = y + intercept
    theta_star = np.insert(theta_star, 0, intercept) #this is the "real theta"
    X = np.column_stack([np.ones(n_samples), X])
    n,p = X.shape
    return X, y

#### Implement ordinary least squares with normal equation

In [23]:
# Fitting an OLS model using normal equations
def ordinary_OLS(X, y):
  theta_hat = np.linalg.inv(X.T @ X) @ X.T @ y

  # Compute training residuals
  y_hat = X @ theta_hat
  residuals = y - y_hat
  traditional_residuals = np.array(residuals)
  return traditional_residuals

#### Implement LOOCV with naive approach

The naive approach consists in doing n iterations. In each iteration i, we remove the row Xi and Yi and compute theta_hat. The residual is obtained with the Yi and Xi that were removed, as if they were as test.

In [24]:
# Implementing naive leave-one-out cross-validation (LOOCV)
def naive_loo(X, y):
  naive_loo_residuals = []
  for i in range(X.shape[0]):
    # remove Xi from the observations
    X_loo = np.delete(X, i, axis=0)
    y_loo = np.delete(y, i, axis=0)

    # Fitting an OLS model using normal equations
    theta_hat = np.linalg.inv(X_loo.T @ X_loo) @ X_loo.T @ y_loo

    # Compute testing residuals
    y_loo_hat = X[i] @ theta_hat
    residual = y[i] - y_loo_hat
    naive_loo_residuals.append(residual)
  return naive_loo_residuals

#### Efficient LOOCV
Here we can use the equation residual = traditional_residual[i]/(1 - H[i][i]) to efficiently compute the new residuals without running n additional iterations

In [25]:
# eficient LOOCV
def efficient_loo(traditional_residuals):
  efficient_loo_residuals = []
  H = X @ (np.linalg.inv(X.T @ X)) @ X.T
  for i in range(X.shape[0]):
    residual = traditional_residuals[i]/ (1 - H[i][i])
    efficient_loo_residuals.append(residual)
  return efficient_loo_residuals

#### Check if the residuals from naive and efficient approach match

In [26]:
#chek if they are equal
X,y = gen_data(n_samples=50, n_features=10, noise=5)
traditional_residuals = ordinary_OLS(X,y)
naive_loo_residuals = naive_loo(X, y)
efficient_loo_residuals = efficient_loo(traditional_residuals)
print(efficient_loo_residuals)
print(naive_loo_residuals)
# np.equal(efficient_loo_residuals, naive_loo_residuals)

[np.float64(-3.2215472387098294), np.float64(1.3541085050115425), np.float64(-2.8951596370763593), np.float64(0.08024375897537439), np.float64(-3.8917544535070205), np.float64(2.332366898987429), np.float64(4.731597553674156), np.float64(6.255198521378412), np.float64(1.1066755606911574), np.float64(6.873582913575118), np.float64(5.781943822580042), np.float64(-7.401078054162488), np.float64(-3.2154452331232495), np.float64(9.610020800851785), np.float64(1.6260396625202351), np.float64(-7.301901282055723), np.float64(0.15209762568970214), np.float64(0.1674933589754648), np.float64(-4.14563006304309), np.float64(2.5611163829592143), np.float64(0.4351334172426647), np.float64(6.282983263722751), np.float64(-4.6565948013912415), np.float64(-1.7043449364399796), np.float64(-2.6354242832665373), np.float64(-1.3396769756113147), np.float64(2.4233389227334117), np.float64(-0.6366939623484508), np.float64(-15.382675093020422), np.float64(-2.505797676135326), np.float64(-9.958427468999599), np.